[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# Learning Rate and Trees


## What you will be able to do

Choose a learning rate and a number of trees together, from the validation loss recorded after
every round, and predict from the first trees of a booster or add more trees to it.

## The idea

### The problem

The **Why Gradient Boosting** notebook added a tenth of every tree, and 300 trees did well. Nothing
said why a tenth, or why 300. A larger share would get there in fewer trees and a smaller one in
more, and the right number of trees depends on the share. So the two settings are one decision,
and a model with a good learning rate and the wrong number of trees can be as poor as one with a bad
rate.

Training a model for every pair of settings would cost a great deal, and one model for each learning
rate is enough, since a booster with 2,000 trees contains the boosters with 1, 2 and 1,999 trees as
its first trees. The validation loss after every round shows how each of them would have done.

### A formal definition

> The **learning rate**, also called **shrinkage**, is the share of each new tree's prediction that is
> added to the model. The **number of rounds**, `n_estimators` in the scikit-learn classes and
> `num_boost_round` in `train`, is how many trees are added. A smaller learning rate needs more
> rounds to reach the same loss.

### Why it works that way

A tree is fitted to the errors that are left, and its answer carries the noise of the rows it was
fitted to along with what is true of every station. With a learning rate of 1.0 the model adds that
whole answer, and later trees build on it, so a model of full steps stops improving early and at a
high loss. With a small rate each tree corrects a little, many trees share the work of every correction, and
the validation loss falls slowly, for hundreds of rounds, to a lower point.

After its lowest point the validation loss rises slowly, because each extra tree fits the training
stations a little more closely, noise included. So each rate has a best number of trees, which the
recorded loss shows, and the lowest losses at different small rates are close
enough that a single validation split cannot always tell them apart.

The libraries let you use the record without training again. A booster can predict from its first
`k` trees, and can be given more trees later.

### Where this shows up

The **Early Stopping** notebook stops training at the best round by itself, and the **Tuning**
notebook searches over settings with cross-validation.

### What this notebook covers

- the validation loss after every round, at five learning rates, in both libraries
- full steps against partial ones
- the best round at each rate, and the test stations measured once for the choice
- predictions from the first trees of a booster
- more trees added to a trained booster
- four mistakes with the rate and the rounds

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np
import lightgbm as lgb

x = np.linspace(0, 6, 200).reshape(-1, 1)
y = np.sin(x.ravel())
for rate in (1.0, 0.1):
    params = {"objective": "regression", "learning_rate": rate, "min_data_in_leaf": 5, "verbose": -1}
    booster = lgb.train(params, lgb.Dataset(x, label=y), num_boost_round=100)
    print(rate, [round(float(((booster.predict(x, num_iteration=k) - y) ** 2).mean()), 4) for k in (1, 10, 100)])
```

```
1.0 [0.0024, 0.0005, 0.0005]
0.1 [0.4216, 0.0641, 0.0005]
```

Each line is a learning rate and the mean squared error after 1, 10 and 100 trees. At 1.0 the first
tree alone brings the error to 0.0024, and ten trees to 0.0005. At 0.1 the same curve takes 100 trees
to get there. These errors are measured on the points the trees were fitted to, so from Setup on the
notebook measures the validation stations instead.

## Setup

Two libraries for the table, the two boosting libraries, and a score.

- `numpy` and `pandas` build the table, and `numpy` finds the best rounds
- `lightgbm` and `xgboost` are the two libraries, and the cell prints their versions
- `mean_squared_error` scores the models on the training, validation and test stations

`station_days` makes the same 20,000 made-up station-days as in the **Why Gradient Boosting**
notebook. Each row is one day at one of 400 weather stations in Norway, with the station's latitude,
elevation, distance from the coast and terrain, the day's cloud, wind and snow, and `tmin`, the day's
lowest temperature. A generator started from a fixed seed draws them, so they are the same on every
run. The cell splits the stations into 300 for training, 50 for validation and 50 for testing,
keeping each station's days together. `FEATURES` names the eight columns a model is given.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


## Worked examples

### LightGBM's validation loss at five learning rates

The training and validation stations go into `Dataset`s once, and `lightgbm_curve` trains 2,000 trees
at one learning rate and returns the validation `l2`, the mean squared error, after each round.

In [2]:
y_train, y_valid, y_test = train["tmin"], valid["tmin"], test["tmin"]
train_dataset = lgb.Dataset(train[FEATURES], label=y_train, free_raw_data=False)
valid_dataset = lgb.Dataset(valid[FEATURES], label=y_valid, reference=train_dataset)


def lightgbm_curve(rate, rounds=2000):
    losses = {}
    booster = lgb.train({"objective": "regression", "learning_rate": rate, "seed": 0, "verbose": -1},
                        train_dataset, rounds, valid_sets=[valid_dataset], valid_names=["valid"],
                        callbacks=[lgb.record_evaluation(losses)])
    return booster, np.array(losses["valid"]["l2"])


RATES = [1.0, 0.3, 0.1, 0.03, 0.01]
MARKS = [10, 30, 100, 300, 1000, 2000]
lightgbm_boosters, lightgbm_curves = {}, {}
for rate in RATES:
    lightgbm_boosters[rate], lightgbm_curves[rate] = lightgbm_curve(rate)

print("rate  " + " ".join(f"{mark:>7}" for mark in MARKS) + "   best round  best loss")
for rate, curve in lightgbm_curves.items():
    best = int(curve.argmin())
    print(f"{rate:<5} " + " ".join(f"{curve[mark - 1]:7.3f}" for mark in MARKS) + f"   {best + 1:10}  {curve[best]:9.3f}")

rate       10      30     100     300    1000    2000   best round  best loss
1.0     4.357   4.455   4.608   5.212   5.426   5.432           17      4.157
0.3     3.338   2.759   2.763   2.883   3.039   3.094           53      2.722
0.1    10.156   3.425   2.553   2.580   2.678   2.752          120      2.546
0.03   26.839  11.833   3.495   2.462   2.438   2.486         1130      2.434
0.01   37.217  26.946  10.711   3.480   2.477   2.481         1189      2.468


At a learning rate of 1.0 the validation loss is lowest after 17 rounds, at 4.157, and climbs to
5.432 by round 2,000. The rates below 0.3 start more slowly, so that after 10 rounds 0.1, 0.03 and
0.01 are still at 10.156, 26.839 and 37.217, and each smaller rate ends lower than the one before it
at round 2,000. The best rounds are 53 at 0.3, 120 at 0.1, 1,130 at 0.03 and 1,189 at 0.01,
with best losses of 2.722, 2.546, 2.434 and 2.468, and past its best round each curve rises again,
slowly.

### The same five rates in XGBoost

XGBoost records `rmse`, the square root of the mean squared error, so `xgboost_curve` squares it to
compare with LightGBM's `l2`.

In [3]:
dtrain = xgb.DMatrix(train[FEATURES], label=y_train)
dvalid = xgb.DMatrix(valid[FEATURES], label=y_valid)
dtest = xgb.DMatrix(test[FEATURES], label=y_test)


def xgboost_curve(rate, rounds=2000):
    losses = {}
    booster = xgb.train({"objective": "reg:squarederror", "learning_rate": rate, "seed": 0}, dtrain, rounds,
                        evals=[(dvalid, "valid")], evals_result=losses, verbose_eval=False)
    return booster, np.array(losses["valid"]["rmse"]) ** 2


xgboost_boosters, xgboost_curves = {}, {}
for rate in RATES:
    xgboost_boosters[rate], xgboost_curves[rate] = xgboost_curve(rate)

print("rate  " + " ".join(f"{mark:>7}" for mark in MARKS) + "   best round  best loss")
for rate, curve in xgboost_curves.items():
    best = int(curve.argmin())
    print(f"{rate:<5} " + " ".join(f"{curve[mark - 1]:7.3f}" for mark in MARKS) + f"   {best + 1:10}  {curve[best]:9.3f}")

rate       10      30     100     300    1000    2000   best round  best loss
1.0     4.188   4.153   4.806   5.092   5.139   5.140           20      4.111
0.3     3.419   2.904   2.924   3.023   3.156   3.198           26      2.881
0.1    10.158   3.316   2.544   2.531   2.613   2.671           77      2.526
0.03   26.815  11.668   3.321   2.509   2.505   2.566          389      2.493
0.01   37.183  26.954  10.521   3.355   2.510   2.533          914      2.506


XGBoost's curves fall and then rise as LightGBM's do, with their lowest points at other rounds. Its
best rounds for the five rates are 20, 26, 77, 389 and 914, and its best losses 4.111, 2.881, 2.526,
2.493 and 2.506. The two libraries' defaults grow trees of different sizes, and XGBoost's penalizes
leaf values with a `lambda` of 1, so one learning rate does not need the same number of rounds in
both.

### Full steps against partial ones

A learning rate of 1.0 adds every tree whole. The training loss shows how closely each rate fits the
training stations by its own best round.

In [4]:
for rate in (1.0, 0.1):
    best = int(lightgbm_curves[rate].argmin()) + 1
    train_loss = mean_squared_error(y_train, lightgbm_boosters[rate].predict(train[FEATURES], num_iteration=best))
    print(f"rate {rate}: best round {best}, training loss {train_loss:.3f}, validation loss {lightgbm_curves[rate][best - 1]:.3f}")

rate 1.0: best round 17, training loss 2.031, validation loss 4.157
rate 0.1: best round 120, training loss 1.361, validation loss 2.546


At its best round the model of full steps has a training loss of 2.031 and a validation loss of
4.157. The model at 0.1 reaches 1.361 and 2.546, so partial steps do better on both sets of stations
with the same kind of tree, which is the comparison the **Why Gradient Boosting** notebook left to
this one.

### Choosing a rate and a number of trees

The choice is made on the validation stations, from the tables above, and the test stations measure
the chosen booster once. `num_iteration` makes a LightGBM booster predict from its first trees, and
`iteration_range=(0, k)` does the same for XGBoost.

In [5]:
for name, curves, boosters in [("LightGBM", lightgbm_curves, lightgbm_boosters),
                               ("XGBoost", xgboost_curves, xgboost_boosters)]:
    rate = min(curves, key=lambda r: curves[r].min())
    best = int(curves[rate].argmin()) + 1
    if name == "LightGBM":
        predictions = boosters[rate].predict(test[FEATURES], num_iteration=best)
    else:
        predictions = boosters[rate].predict(dtest, iteration_range=(0, best))
    print(f"{name}: rate {rate}, {best} trees, validation {curves[rate][best - 1]:.3f}, "
          f"test {mean_squared_error(y_test, predictions):.3f}")

LightGBM: rate 0.03, 1130 trees, validation 2.434, test 2.528
XGBoost: rate 0.03, 389 trees, validation 2.493, test 2.530


Both libraries' lowest validation loss comes at a rate of 0.03, after 1,130 trees in LightGBM and 389
in XGBoost, and the test stations, measured once, give 2.528 and 2.530. The best losses at 0.1 and at
0.01 are within 0.12 of 0.03's in both libraries, so another split of the stations could choose
differently.

### Predicting from the first trees

One booster of 2,000 trees answers for any smaller number of them. The losses below are the ones the
recorded curve holds for those rounds.

In [6]:
booster = lightgbm_boosters[0.1]
for trees in (10, 100, 1000):
    loss = mean_squared_error(y_valid, booster.predict(valid[FEATURES], num_iteration=trees))
    print(trees, "trees:", round(loss, 3), "| recorded:", round(lightgbm_curves[0.1][trees - 1], 3))

10 trees: 10.156 | recorded: 10.156
100 trees: 2.553 | recorded: 2.553
1000 trees: 2.678 | recorded: 2.678


The losses match the recorded curve, because the curve was computed from these same trees as they
were added.

### More trees for a trained booster

`init_model` gives `lgb.train` a booster to start from, and `xgb_model` does the same for `xgb.train`.
The new trees fit the errors the old ones left. LightGBM stores the old booster's predictions in the
`Dataset` as the starting point of every booster trained from it afterward, so the cell gives the
continued booster a `Dataset` of its own. `free_raw_data=False` keeps the table's values, from which
LightGBM computes those predictions once the `Dataset` is built.

In [7]:
continued_dataset = lgb.Dataset(train[FEATURES], label=y_train, free_raw_data=False)
short_lightgbm = lgb.train({"objective": "regression", "learning_rate": 0.1, "seed": 0, "verbose": -1},
                           continued_dataset, 50)
longer_lightgbm = lgb.train({"objective": "regression", "learning_rate": 0.1, "seed": 0, "verbose": -1},
                            continued_dataset, 50, init_model=short_lightgbm)
short_xgboost = xgb.train({"objective": "reg:squarederror", "learning_rate": 0.1, "seed": 0}, dtrain, 50)
longer_xgboost = xgb.train({"objective": "reg:squarederror", "learning_rate": 0.1, "seed": 0}, dtrain, 50,
                           xgb_model=short_xgboost)
for name, short, longer, data in [("LightGBM", short_lightgbm, longer_lightgbm, valid[FEATURES]),
                                  ("XGBoost", short_xgboost, longer_xgboost, dvalid)]:
    trees = short.num_trees() if name == "LightGBM" else short.num_boosted_rounds()
    more = longer.num_trees() if name == "LightGBM" else longer.num_boosted_rounds()
    print(f"{name}: {trees} trees, validation {mean_squared_error(y_valid, short.predict(data)):.3f} | "
          f"{more} trees, validation {mean_squared_error(y_valid, longer.predict(data)):.3f}")

LightGBM: 50 trees, validation 2.724 | 100 trees, validation 2.553
XGBoost: 50 trees, validation 2.638 | 100 trees, validation 2.544


Each continued booster has 100 trees, and its validation loss, 2.553 for LightGBM and 2.544 for
XGBoost, is the loss the recorded curves show at round 100 for a rate of 0.1, the same as training
in one run of 100 rounds, to the three places printed.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/04-learning-rate-and-trees-solutions.ipynb).

**1.** Record LightGBM's validation loss at a learning rate of 0.05 for 2,000 rounds with `lightgbm_curve`, and print its best round and best loss.

In [8]:
# your code here

**2.** Do the same for XGBoost with `xgboost_curve` at 0.05.

In [9]:
# your code here

**3.** Take the LightGBM booster at a learning rate of 0.03 from `lightgbm_boosters`, and print its validation loss from its first 50, 500 and 1,500 trees.

In [10]:
# your code here

**4.** Train an XGBoost booster for 100 rounds at 0.1, then give it 200 more with `xgb_model`, and print the validation loss after 100 and after 300 rounds.

In [11]:
# your code here

**5.** Fit an `XGBRegressor` with the learning rate XGBoost chose in the worked examples and `n_estimators` set to its best round, and print its test loss beside the booster's.

In [12]:
# your code here

## Common errors

### No error, and learning rates compared at the same number of trees

Every rate gets 100 trees, as if the number of trees were a fixed part of the model.

In [13]:
for rate in RATES:
    print(f"rate {rate}: validation loss after 100 trees {lightgbm_curves[rate][99]:.3f}, "
          f"lowest {lightgbm_curves[rate].min():.3f}")

rate 1.0: validation loss after 100 trees 4.608, lowest 4.157
rate 0.3: validation loss after 100 trees 2.763, lowest 2.722
rate 0.1: validation loss after 100 trees 2.553, lowest 2.546
rate 0.03: validation loss after 100 trees 3.495, lowest 2.434
rate 0.01: validation loss after 100 trees 10.711, lowest 2.468


At 100 trees the rate of 0.1 looks best, and 0.03 and 0.01 look poor, at 3.495 and 10.711. Each at
its own best round, they reach 2.434 and 2.468, below 0.1's 2.546 on these validation stations, by
gaps the worked examples found small enough for another split to reverse. Compare learning rates each
at its best round.

### `XGBoostError: Check failed: end <= model.BoostedRounds() (500 vs. 100) : Out of range for tree layers.`

A booster of 100 trees is asked for the predictions of its first 500.

In [14]:
hundred = xgb.train({"objective": "reg:squarederror", "learning_rate": 0.1, "seed": 0}, dtrain, 100)
try:
    hundred.predict(dvalid, iteration_range=(0, 500))
except xgb.core.XGBoostError as error:
    first_line = str(error).splitlines()[0]
    print("XGBoostError:", first_line[first_line.index("Check failed"):])

XGBoostError: Check failed: end <= model.BoostedRounds() (500 vs. 100) : Out of range for tree layers.


`iteration_range=(0, 500)` asks for the first 500 trees of a booster that has 100. XGBoost raises an
`XGBoostError` whose message starts with a time and a path inside its build, which the cell leaves
out. Ask for at most `num_boosted_rounds()` trees.

### No error, and a LightGBM prediction from 100 trees where 500 were asked for

In [15]:
hundred_lightgbm = lgb.train({"objective": "regression", "learning_rate": 0.1, "seed": 0, "verbose": -1},
                             train_dataset, 100)
print(hundred_lightgbm.num_trees(), "trees")
print(np.array_equal(hundred_lightgbm.predict(valid[FEATURES], num_iteration=500),
                     hundred_lightgbm.predict(valid[FEATURES])))

100 trees
True


Asked for 500 trees, LightGBM used all 100 it has, and the predictions equal the whole booster's, so
a wrong number of trees passes without a word. Check `num_trees()` before asking for some of them.

### No error, and a second fit that starts again from the first tree

In [16]:
model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=0)
model.fit(train[FEATURES], y_train)
model.fit(train[FEATURES], y_train)
print(model.get_booster().num_boosted_rounds(), "trees after two fits")

100 trees after two fits


`fit` trains a new model every time it is called, so the second call trained 100 trees again in
place of the first 100. To add trees, train with `xgb_model` or `init_model`, as the worked examples
do, or set `n_estimators` to the total you want.

## Recap

- You can record the validation loss after every round and read the best number of trees for a
  learning rate from it.
- You can tell full steps from partial ones by where the validation loss stops falling and how low
  it gets.
- You can choose the rate and the rounds on the validation stations, and measure the test stations
  once.
- You can predict from the first trees of a booster and add trees to a trained one.

## What is next

The **Early Stopping** notebook stops adding trees once the validation loss has stopped improving,
so the best round no longer has to be read off a table.

---

&#8592; **Previous:** [The Native Interfaces](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/03-the-native-interfaces.ipynb)  &nbsp;·&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Early Stopping](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/05-early-stopping.ipynb) &#8594;
